# Step 1: Input sequences and clean

In [1]:
dna_fs = input("Enter the fusion sequence: ")

gene_a_cds = input("Enter Gene A CDS sequence: ")

gene_b_cds = input("Enter Gene B CDS sequence: ")

# Optional: the frame the fusion caller (Arriba/FusionCatcher) predicted.
# Leave blank if you are running this standalone, without a TSV row.
reported_frame = input(
    "Enter reported/predicted effect from fusion caller "
    "(in-frame / out-of-frame, leave blank if unknown): "
).strip()

if reported_frame == "":
    reported_frame = None


gene_a = gene_a_cds.replace(" ", "").upper()
gene_b = gene_b_cds.replace(" ", "").upper()
dna_fs = dna_fs.replace(" ", "").upper()


Enter the fusion sequence:  CGCCAAGTGGCCGTACCGCGGCGGCGGCGGCGGCGGCAGCGCGGGGGGCG*GCGGGCAGTTCGGCGGCGCGGGGCCCGGGGCCGGGGGCGGCGGCGGCCCC
Enter Gene A CDS sequence:  atg aacttccagg cgggcggggg gcagagcccg cagcagcagc agagcctggc ggctccgggg ggcggcggcg ctgccgcgca gcagctcgtc tgcggcgggc agttcggcgg cgcggggccc ggggccgggg gcggcggcgg cccctcgcag cagctggccg gcgggccccc ccagcagttc gcgctctcca actccgcggc catccgggcc gagatccagc gcttcgagtc cgtgcatccc aatatctacg ccatctacga cctgatcgag cgcatcgagg atttggcgct gcagaaccag atccgggagc acgtcatctc catcgaggac tcgtttgtga acagccagga gtggacgctg agccgctccg taccggagct taaagtgggc atagtgggga acctgtctag cgggaagtca gccctggtgc accgctatct gacggggacc tatgtccagg aggagtcccc tgaagggggg cggtttaaga aggagattgt ggtggatggc cagagttacc tgctgctgat ccgagatgaa ggaggccccc ctgagctcca gtttgctgcc tgggtggatg cagtggtgtt tgtgttcagc ctggaggatg aaatcagttt ccagacggtg tacaactact tcctgcgtct ctgcagcttc cgcaacgcca gcgaggtgcc catggtgctt gtgggcacgc aggatgccat cagcgctgcg aatccccggg ttatcgacga cagcagagcc cgcaagctct cc

# Step 2: Split the fusion sequence and search for each fragment

In [2]:
left_fragment, right_fragment = dna_fs.split("*")


def search(fragment, gene):

    position = gene.find(fragment)

    if position == -1:
        return "Not found"

    return position


print("\nLEFT FRAGMENT")
print("Gene A:", search(left_fragment, gene_a))
print("Gene B:", search(left_fragment, gene_b))

print("\nRIGHT FRAGMENT")
print("Gene A:", search(right_fragment, gene_a))
print("Gene B:", search(right_fragment, gene_b))


left_in_a = gene_a.find(left_fragment)
left_in_b = gene_b.find(left_fragment)

right_in_a = gene_a.find(right_fragment)
right_in_b = gene_b.find(right_fragment)


LEFT FRAGMENT
Gene A: Not found
Gene B: 209

RIGHT FRAGMENT
Gene A: 97
Gene B: Not found


# Step 3: Reconstruct the fusion cds, save junction

In [3]:
# Track which gene is upstream (5') and which is downstream (3') of junction, plus downstream gene's own native index at the breakpoint (downstream_index). 
# This is the number the frame-status check needs, and it works no matter which gene ends up on which side.

if left_in_a != -1 and right_in_b != -1:

    left_sequence = gene_a[:left_in_a + len(left_fragment)]
    right_sequence = gene_b[right_in_b:]

    upstream_gene_name = "Gene A"
    downstream_gene_name = "Gene B"
    downstream_index = right_in_b

elif left_in_b != -1 and right_in_a != -1:

    left_sequence = gene_b[:left_in_b + len(left_fragment)]
    right_sequence = gene_a[right_in_a:]

    upstream_gene_name = "Gene B"
    downstream_gene_name = "Gene A"
    downstream_index = right_in_a


else:

    print("\nUnable to reconstruct the fusion CDS.")
    exit()


fusion_cds = left_sequence + right_sequence

print("\nFusion CDS:\n")
print(fusion_cds)

print("\nUpstream gene (5' of junction):", upstream_gene_name)
print("Downstream gene (3' of junction):", downstream_gene_name)

junction = len(left_sequence)

print("\nDNA junction position:", junction)
print("Reading frame remainder (junction mod 3):", junction % 3)




Fusion CDS:

ATGAACGACTTTGACGAGTGCGGCCAGAGCGCAGCCAGCATGTACCTGCCGGGCTGCGCCTACTATGTGGCCCCGTCTGACTTCGCTAGCAAGCCTTCGTTCCTTTCCCAACCGTCGTCCTGCCAGATGACTTTCCCCTACTCTTCCAACCTGGCTCCGCACGTCCAGCCCGTGCGCGAAGTGGCCTTCCGCGACTACGGCCTGGAGCGCGCCAAGTGGCCGTACCGCGGCGGCGGCGGCGGCGGCAGCGCGGGGGGCGGCGGGCAGTTCGGCGGCGCGGGGCCCGGGGCCGGGGGCGGCGGCGGCCCCTCGCAGCAGCTGGCCGGCGGGCCCCCCCAGCAGTTCGCGCTCTCCAACTCCGCGGCCATCCGGGCCGAGATCCAGCGCTTCGAGTCCGTGCATCCCAATATCTACGCCATCTACGACCTGATCGAGCGCATCGAGGATTTGGCGCTGCAGAACCAGATCCGGGAGCACGTCATCTCCATCGAGGACTCGTTTGTGAACAGCCAGGAGTGGACGCTGAGCCGCTCCGTACCGGAGCTTAAAGTGGGCATAGTGGGGAACCTGTCTAGCGGGAAGTCAGCCCTGGTGCACCGCTATCTGACGGGGACCTATGTCCAGGAGGAGTCCCCTGAAGGGGGGCGGTTTAAGAAGGAGATTGTGGTGGATGGCCAGAGTTACCTGCTGCTGATCCGAGATGAAGGAGGCCCCCCTGAGCTCCAGTTTGCTGCCTGGGTGGATGCAGTGGTGTTTGTGTTCAGCCTGGAGGATGAAATCAGTTTCCAGACGGTGTACAACTACTTCCTGCGTCTCTGCAGCTTCCGCAACGCCAGCGAGGTGCCCATGGTGCTTGTGGGCACGCAGGATGCCATCAGCGCTGCGAATCCCCGGGTTATCGACGACAGCAGAGCCCGCAAGCTCTCCACAGATCTGAAGCGGTGCACCTACTATGAGACGTGCGCGACCTACGGGCTCAATGTGGA

# Step 4: Translate fusion CDS and both native CDS

In [4]:
# Translate() stops at the first stop codon, matching real ribosome behaviour. 
# We need this not only for the fusion transcript but also for both native genes, since the native protein sequences are required for the empirical novelty
# check in Step 7.

genetic_code = {

    "ATA":"I","ATC":"I","ATT":"I","ATG":"M",
    "ACA":"T","ACC":"T","ACG":"T","ACT":"T",
    "AAC":"N","AAT":"N","AAA":"K","AAG":"K",
    "AGC":"S","AGT":"S","AGA":"R","AGG":"R",
    "CTA":"L","CTC":"L","CTG":"L","CTT":"L",
    "CCA":"P","CCC":"P","CCG":"P","CCT":"P",
    "CAC":"H","CAT":"H","CAA":"Q","CAG":"Q",
    "CGA":"R","CGC":"R","CGG":"R","CGT":"R",
    "GTA":"V","GTC":"V","GTG":"V","GTT":"V",
    "GCA":"A","GCC":"A","GCG":"A","GCT":"A",
    "GAC":"D","GAT":"D","GAA":"E","GAG":"E",
    "GGA":"G","GGC":"G","GGG":"G","GGT":"G",
    "TCA":"S","TCC":"S","TCG":"S","TCT":"S",
    "TTC":"F","TTT":"F","TTA":"L","TTG":"L",
    "TAC":"Y","TAT":"Y","TAA":"*","TAG":"*","TGA":"*"

}

def translate(dna):

    protein = ""

    for i in range(0, len(dna) - 2, 3):

        codon = dna[i:i + 3]

        amino_acid = genetic_code.get(codon, "X")

        if amino_acid == "*":
            break

        protein += amino_acid

    return protein


protein = translate(fusion_cds)
gene_a_protein = translate(gene_a_cds)
gene_b_protein = translate(gene_b_cds)

junction_aa = junction // 3

print("\nFusion protein (stops at first stop codon):\n")
print(protein)

print("\nAA junction position:", junction_aa)

if junction_aa < len(protein):
    print("AA at junction:", protein[junction_aa])
else:
    print("Junction falls beyond the translated protein "
          "(an early stop codon was hit before reaching it).")




Fusion protein (stops at first stop codon):

MNDFDEXGQSAASMYLPGXAYYVAPSDFASKPSFLSQPSSXQMTFPYSSNLAPHVQPVREVAFRDYGLERAKXPYRGGGGGGSAGGGGQFGGAGPGAGGGGGPSQQLAGGPPQQFALSNSAAIRAEIQRFESVHPNIYAIYDLIERIEDLALQNQIREHVISIEDSFVNSQEXTLSRSVPELKVGIVGNLSSGKSALVHRYLTGTYVQEESPEGGRFKKEIVVDGQSYLLLIRDEGGPPELQFAAXVDAVVFVFSLEDEISFQTVYNYFLRLXSFRNASEVPMVLVGTQDAISAANPRVIDDSRARKLSTDLKRXTYYETXATYGLNVERVFQDVAQKVVALRKKQQLAIGPXKSLPNSPSHSAVSAASIPAVHINQATNGGGSAFSDYSSSVPSTPSISQRELRIETIAASSTPTPIRKQSKRRSNIFTSRKGADLDREKKAAEXKVDSIGSGRAIPIKQGILLKRSGKSLNKEXKKKYVTLXDNGLLTYHPSLHDYMQNIHGKEIDLLRTTVKVPGKRLPRATPATAPGTSPRANGLSVERSNTQLGGGTGAPHSASSASLHSERPLSSSAXAGPRPEGLHQRSXSVSSADQXSEATTSLPPGMQHPASGPAEVLSSSPKLDPPPSPHSNRKKHRRKKSTGTPRPDGPSSATEEAEESFEFVVVSLTGQTXHFEASTAEERELXVQSVQAQILASLQGXRSAKDKTRLGNQNAALAVQAVRTVRGNSFXIDXDAPNPDXASLNLGALMXIEXSGIHRHLGAHLSRVRSLDLDDXPPELLAVMTAMGNALANSVXEGALGGYSKPGPDAXREEKERXIRAKYEQKLFLAPLPSSDVPLGQQLLRAVVEDDLRLLVMLLAHGSKEEVNETYGDGDGRTALHLSSAMANVVFTQLLIXYGVDVRSRDARGLTPLAYARRAGSQEXADILIQHGXPGEGXGLAPTPNREPANGTNP

# Step 5: Determining frame status with 2 independent methods

In [6]:

# Method 1 (arithmetic): compares the codon-phase of the junction
# within the fusion transcript to the codon-phase of the same
# breakpoint within the downstream gene's OWN native CDS. This is
# the general rule -- it works for in-frame and out-of-frame
# fusions alike:
#
#     junction % 3 == downstream_index % 3   ->  in-frame
#     junction % 3 != downstream_index % 3   ->  out-of-frame
#
# Method 2 (empirical/identity-based): translate the downstream
# portion of the fusion protein and directly compare it, residue
# by residue, against the downstream gene's native protein from
# the corresponding position. High identity means the fusion is
# reading in the native frame; low identity means it has shifted.

if junction % 3 == downstream_index % 3:
    calculated_frame_math = "in-frame"
else:
    calculated_frame_math = "out-of-frame"

print("\nFrame status: Method 1 (arithmetic)")
print("Calculated frame (math):", calculated_frame_math)


# Downstream gene's own native protein, generalized to whichever
# gene ended up downstream:
if downstream_gene_name == "Gene B":
    downstream_native_protein = gene_b_protein
else:
    downstream_native_protein = gene_a_protein

# Amino acids the fusion actually produced downstream of the junction:
fusion_downstream_protein = protein[junction_aa:]

# Where the breakpoint falls in the downstream gene's own AA coordinates:
downstream_junction_aa = downstream_index // 3

# Native amino acids from that same position onward:
native_downstream_protein = downstream_native_protein[downstream_junction_aa:]

comparison_length = min(30, len(fusion_downstream_protein), len(native_downstream_protein))

if comparison_length > 0:

    matches = 0

    for i in range(comparison_length):
        if fusion_downstream_protein[i] == native_downstream_protein[i]:
            matches += 1

    identity = matches / comparison_length

else:
    identity = 0.0

print("\nFrame status: Method 2 (empirical identity)")
print("Compared", comparison_length, "residues downstream of the junction.")
print("Identity to native", downstream_gene_name + ":", round(identity * 100, 2), "%")

if identity >= 0.90:
    calculated_frame_identity = "in-frame"
else:
    calculated_frame_identity = "out-of-frame"

print("Calculated frame (identity):", calculated_frame_identity)


# Cross-check the two independent frame-status methods against each other:
frame_methods_agree = (calculated_frame_math == calculated_frame_identity)

print("\nFrame status cross-check")
if frame_methods_agree:
    print("Both methods agree:", calculated_frame_math)
else:
    print("WARNING: the two frame-status methods disagree.")
    print("Math method says:", calculated_frame_math)
    print("Identity method says:", calculated_frame_identity)
    print("Investigate before trusting downstream peptide calls.")

# The math-based result is used as the deterministic driver for peptide
# selection logic below, since it is exact rather than threshold-based.
calculated_frame = calculated_frame_math

# Optional comparison against what the fusion caller itself reported:
if reported_frame is not None:

    print("\nComparison with fusion caller")
    print("Reported:", reported_frame)
    print("Calculated:", calculated_frame)

    if reported_frame.lower() == calculated_frame.lower():
        print("Frame agrees with fusion caller.")
    else:
        print("Frame does NOT agree with fusion caller.")



Frame status: Method 1 (arithmetic)
Calculated frame (math): in-frame

Frame status: Method 2 (empirical identity)
Compared 30 residues downstream of the junction.
Identity to native Gene A: 0.0 %
Calculated frame (identity): out-of-frame

Frame status cross-check
Math method says: in-frame
Identity method says: out-of-frame
Investigate before trusting downstream peptide calls.

Comparison with fusion caller
Reported: in-frame
Calculated: in-frame
Frame agrees with fusion caller.


# Step 6: Generate Candidate Peptides by 3 independent methods

In [7]:
# For in-frame fusions: only peptides whose window overlaps the single junction residue are candidates.
# For out-of-frame fusions: peptides overlapping the junction + every peptide fully downstream of it are candidates, since the
# entire downstream region is shifted out of the native frame.
# 3 independent  coded methods are generated and cross-checked. If they agree, that is strong evidence the window logic is correct.

window_size = int(input("\nEnter peptide length: "))

if calculated_frame == "in-frame":
    novel_start_aa = junction_aa
    novel_end_aa = junction_aa
else:
    novel_start_aa = junction_aa
    novel_end_aa = len(protein) - 1


Enter peptide length:  8


In [8]:
# Method A: brute-force filter over every window 

all_peptides = []

for start in range(len(protein) - window_size + 1):

    end = start + window_size - 1
    peptide = protein[start:start + window_size]

    all_peptides.append((start, end, peptide))


method_a_peptides = []

for start, end, peptide in all_peptides:

    overlaps_novel_region = (start <= novel_end_aa and end >= novel_start_aa)

    if overlaps_novel_region:
        method_a_peptides.append((start, peptide))

In [9]:
#  Method B: direct computed start range 
earliest_start = max(0, novel_start_aa - window_size + 1)
latest_start = min(len(protein) - window_size, novel_end_aa)

method_b_peptides = []

for start in range(earliest_start, latest_start + 1):

    peptide = protein[start:start + window_size]

    method_b_peptides.append((start, peptide))

#  Method C: offset-from-junction reconstruction 

In [10]:
method_c_peptides = []
seen_positions = set()

for offset in range(window_size):

    start = novel_start_aa - offset

    if start < 0 or start + window_size > len(protein):
        continue

    peptide = protein[start:start + window_size]

    if start not in seen_positions:
        method_c_peptides.append((start, peptide))
        seen_positions.add(start)

if calculated_frame == "out-of-frame":

    for start in range(novel_start_aa + 1, novel_end_aa + 1):

        if start + window_size > len(protein):
            continue

        if start in seen_positions:
            continue

        peptide = protein[start:start + window_size]
        method_c_peptides.append((start, peptide))
        seen_positions.add(start)

method_c_peptides.sort(key=lambda item: item[0])



# Cross Check the 3 methods

In [11]:
set_a = set(method_a_peptides)
set_b = set(method_b_peptides)
set_c = set(method_c_peptides)

print("\n" + "=" * 60)
print("CANDIDATE PEPTIDES (window-generation cross-check)")
print("=" * 60)

print("\nMethod A (brute-force filter):")
for start, peptide in method_a_peptides:
    print(start, peptide)

print("\nMethod B (direct computed range):")
for start, peptide in method_b_peptides:
    print(start, peptide)

print("\nMethod C (offset-from-junction):")
for start, peptide in method_c_peptides:
    print(start, peptide)

if set_a == set_b == set_c:
    print("\nAll three window-generation methods agree:", len(set_a), "candidate peptide(s).")
else:
    print("\nWARNING: window-generation methods do NOT agree. Investigate before trusting output.")
    print("Method A unique:", set_a - (set_b | set_c))
    print("Method B unique:", set_b - (set_a | set_c))
    print("Method C unique:", set_c - (set_a | set_b))

candidate_peptides = method_a_peptides   # any of the three, since they agree

# Split candidates into "junction" (overlaps the junction residue itself)
# vs "downstream" (fully after it, only relevant for out-of-frame fusions),
# matching the two labelled categories used downstream in the CSV output.

junction_peptides = []
downstream_peptides = []

for start, peptide in candidate_peptides:

    end = start + window_size - 1

    if start <= junction_aa <= end:
        junction_peptides.append((start, peptide))
    elif calculated_frame == "out-of-frame" and start > junction_aa:
        downstream_peptides.append((start, peptide))



CANDIDATE PEPTIDES (window-generation cross-check)

Method A (brute-force filter):
79 GGGSAGGG
80 GGSAGGGG
81 GSAGGGGQ
82 SAGGGGQF
83 AGGGGQFG
84 GGGGQFGG
85 GGGQFGGA
86 GGQFGGAG

Method B (direct computed range):
79 GGGSAGGG
80 GGSAGGGG
81 GSAGGGGQ
82 SAGGGGQF
83 AGGGGQFG
84 GGGGQFGG
85 GGGQFGGA
86 GGQFGGAG

Method C (offset-from-junction):
79 GGGSAGGG
80 GGSAGGGG
81 GSAGGGGQ
82 SAGGGGQF
83 AGGGGQFG
84 GGGGQFGG
85 GGGQFGGA
86 GGQFGGAG

All three window-generation methods agree: 8 candidate peptide(s).


# Step 7: Novelty Confirmation

In [12]:
# A second, independent validation axis, orthogonal to the frame math above: a peptide is only confirmed novel if the exact amino acid sequence does NOT already appear anywhere in the
# full native gene A or gene B protein. 
# This directly checks ground truth rather than relying on frame arithmetic, and acts as a safety net against any subtle frame-logic error.

novel_peptides = []

for start, peptide in junction_peptides:

    in_gene_a = peptide in gene_a_protein
    in_gene_b = peptide in gene_b_protein

    if not in_gene_a and not in_gene_b:
        novel_peptides.append((start, peptide, "junction"))

for start, peptide in downstream_peptides:

    in_gene_a = peptide in gene_a_protein
    in_gene_b = peptide in gene_b_protein

    if not in_gene_a and not in_gene_b:
        novel_peptides.append((start, peptide, "downstream"))

print("\n" + "=" * 60)
print("NOVEL PEPTIDES (confirmed absent from both native proteins)")
print("=" * 60)

for start, peptide, kind in novel_peptides:
    print("AA", start, "|", kind, "|", peptide)

print(
    "\n", len(candidate_peptides), "candidate peptide(s) generated;",
    len(novel_peptides), "confirmed novel after the native-protein check."
)




NOVEL PEPTIDES (confirmed absent from both native proteins)
AA 79 | junction | GGGSAGGG
AA 80 | junction | GGSAGGGG
AA 81 | junction | GSAGGGGQ
AA 82 | junction | SAGGGGQF
AA 83 | junction | AGGGGQFG
AA 84 | junction | GGGGQFGG
AA 85 | junction | GGGQFGGA
AA 86 | junction | GGQFGGAG

 8 candidate peptide(s) generated; 8 confirmed novel after the native-protein check.


# Step 8: Annotate and save every peptide

In [13]:
annotated_peptides = []

for start, peptide, kind in novel_peptides:

    annotated_peptides.append({

        "GeneA": gene_a,
        "GeneB": gene_b,
        "Upstream_gene": upstream_gene_name,
        "Downstream_gene": downstream_gene_name,
        "Reported_frame": reported_frame,
        "Calculated_frame_math": calculated_frame_math,
        "Calculated_frame_identity": calculated_frame_identity,
        "Frame_methods_agree": frame_methods_agree,
        "Peptide_type": kind,
        "Peptide": peptide,
        "Peptide_length": len(peptide),
        "AA_position": start,
        "DNA_junction": junction,
        "AA_junction": junction_aa,
        "Fusion_sequence": dna_fs,

    })

print("\nCandidate neoantigen peptides\n")

for peptide_record in annotated_peptides:
    print(peptide_record)



Candidate neoantigen peptides

{'GeneA': 'ATGAACTTCCAGGCGGGCGGGGGGCAGAGCCCGCAGCAGCAGCAGAGCCTGGCGGCTCCGGGGGGCGGCGGCGCTGCCGCGCAGCAGCTCGTCTGCGGCGGGCAGTTCGGCGGCGCGGGGCCCGGGGCCGGGGGCGGCGGCGGCCCCTCGCAGCAGCTGGCCGGCGGGCCCCCCCAGCAGTTCGCGCTCTCCAACTCCGCGGCCATCCGGGCCGAGATCCAGCGCTTCGAGTCCGTGCATCCCAATATCTACGCCATCTACGACCTGATCGAGCGCATCGAGGATTTGGCGCTGCAGAACCAGATCCGGGAGCACGTCATCTCCATCGAGGACTCGTTTGTGAACAGCCAGGAGTGGACGCTGAGCCGCTCCGTACCGGAGCTTAAAGTGGGCATAGTGGGGAACCTGTCTAGCGGGAAGTCAGCCCTGGTGCACCGCTATCTGACGGGGACCTATGTCCAGGAGGAGTCCCCTGAAGGGGGGCGGTTTAAGAAGGAGATTGTGGTGGATGGCCAGAGTTACCTGCTGCTGATCCGAGATGAAGGAGGCCCCCCTGAGCTCCAGTTTGCTGCCTGGGTGGATGCAGTGGTGTTTGTGTTCAGCCTGGAGGATGAAATCAGTTTCCAGACGGTGTACAACTACTTCCTGCGTCTCTGCAGCTTCCGCAACGCCAGCGAGGTGCCCATGGTGCTTGTGGGCACGCAGGATGCCATCAGCGCTGCGAATCCCCGGGTTATCGACGACAGCAGAGCCCGCAAGCTCTCCACAGATCTGAAGCGGTGCACCTACTATGAGACGTGCGCGACCTACGGGCTCAATGTGGAGCGTGTCTTCCAGGACGTGGCCCAGAAGGTAGTGGCCTTGCGAAAGAAGCAGCAACTGGCCATCGGGCCCTGCAAGTCACTGCCCAACTCGCCCAGCCACTCGGCCGTGTCCGCCGCCTCCATCCCGGCCGTG

# Step 9: Export CSV

In [14]:


# For a single standalone run, this writes just this fusion's peptides. When this script is looped over multiple fusions
# (e.g. driven by the TSV + sample metadata), accumulate into "results" across iterations before writing the final CSV once.

import pandas as pd

results = list(annotated_peptides)

results_df = pd.DataFrame(results)

results_df.to_csv("Fusion_candidate_peptides.csv", index=False)

print("\nFinished.")
print("Number of candidate peptides:", len(results_df))
print("Results saved as Fusion_candidate_peptides.csv")


Finished.
Number of candidate peptides: 8
Results saved as Fusion_candidate_peptides.csv
